# Good Word Attack on a Naive Bayes Spam Filter (Black-Box)

**Author:** Teodora Demerdzhieva  
**Topic:** Adversarial Machine Learning - Evasion Attacks (Feature Obfuscation / "Good Words" Attacks)

This is the same attack as in the white-box notebook, but now the model is a black box.

In the white-box version, I could read the weight table (`feature_log_prob_`) and pick the best words straight away. Here I can't do that. The model is just something I send text to, and it sends back a number. To find words that work, I have to try them one by one, and each try costs a query from a limited budget. So this is no longer a simple lookup. It's a search, and I have to decide carefully where to spend each query.

This is also the more realistic case. In most real attacks, you don't get a copy of the model. You get an API behind a web form, and that's it.


### Attack goal

| | |
|---|---|
| Starting messages | Spam |
| Target verdict | Ham |
| Model access | Query API only |
| Query budget | Limited (set by rate limits or cost) |
| Word budget per message | 25 (a typical limit for scoring services) |

## How It Works Without the Weights

The underlying math hasn't changed from the white-box notebook. Naive Bayes just adds up a log-probability for each word. If you add words that strongly favor "ham," the total moves toward "ham." The model has no way to notice that those words don't really belong in the message.

What's different is how we find the good words. In the white-box case, we just read the weight table. Here, we don't get to see any weights. Instead, we add a word to a spam message and check how much the score moved. Do that for a few messages and average it. Each check costs one query. If we had unlimited queries, we could just test every word on every message. Since we don't, we need a smarter way to pick what to test next.

Three ideas help with that. Exploitation means sticking with words that already look good. If `later` has tested well ten times in a row, keep testing `later`. Exploration means trying words we haven't tested much yet, since a word that looks weak on average might actually work well on a particular message. Combination search means testing pairs or small groups of words, to check if two words together work better than the sum of their separate effects. On this model (MNB) they don't, because the model adds things up and nothing more. On other models they might, and that changes how you'd build the final payload.

To explore efficiently, I use an algorithm called UCB1 (Auer, Cesa-Bianchi & Fischer, 2002). Think of each candidate word as a slot machine arm. Each time we test a word, that's one pull, and the probability drop we measure is the reward. UCB gives each word a score: mean_reward + c·sqrt(ln(t) / n). The first part rewards words that have worked well so far. The second part is a bonus for words we haven't tested much, and that bonus shrinks the more we test a word. A word we've never tried gets an infinite score, so it always gets tried at least once. The `c` setting controls how much the algorithm favors the current best word versus trying new ones. c=2.0 is the standard default.

One thing worth knowing: the reward here is measured as a probability drop, not a log-odds drop. Because of how the model actually works internally, the same word can look like it causes a bigger drop on a message that starts near 50% than on one that starts at 99.9%, just because of where that message happens to sit on the curve. This doesn't break anything in this notebook, since every word gets tested against the same shared set of messages. But it's part of why UCB's top list and the full scan's top list below don't match perfectly.

## Setup

In [1]:
import html
import os
import random
from itertools import combinations

import numpy as np
import pandas as pd
import requests
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

SEED = 1337
random.seed(SEED)
np.random.seed(SEED)

pd.set_option("display.max_colwidth", None)
pd.options.mode.string_storage = "python"
print("Setup complete.")

Setup complete.


## Dataset

Same SMS Spam Collection as the white-box notebook, but this time two batches are held out of training. The six attack targets are the same six as before, so the two notebooks can be compared directly. The fifteen discovery messages are new: the black-box search uses them to measure each word's effect, while keeping the attack targets untouched.

Both batches sit outside the training set, so the model's scores on them reflect what it actually learned, not what it just memorized.

*The dataset is saved locally at `data/sms.tsv` next to this notebook, so results stay reproducible even if the original GitHub file changes later.*

In [2]:
DATA_PATH = "data/sms.tsv"
DATA_URL = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"

if os.path.exists(DATA_PATH):
    sms = pd.read_csv(DATA_PATH, sep="\t", header=None, names=["label", "text"])
else:
    # Fallback for environments without the vendored copy. Note this pulls
    # from a moving branch, not a pinned commit, so counts below may drift
    # slightly from what's printed in this notebook if the upstream file
    # ever changes.
    sms = pd.read_csv(DATA_URL, sep="\t", header=None, names=["label", "text"])
sms["text"] = sms["text"].apply(html.unescape)

TARGET_SPAM = [
    "Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's",
    "U have won a nokia 6230 plus a free digital camera. This is what u get when u win our FREE auction. To take part send NOKIA to 83383 now. POBOX114/14TCR/W1 16",
    "Thanks for your ringtone order, reference number X29. Your mobile will be charged 4.50. Should your tone not arrive please call customer services 09065989180",
    "Want a new Video Phone? 750 anytime any network mins? Half price line rental free text for 3 months? Reply or call 08000930705 for free delivery",
    "We know someone who you know that fancies you. Call 09058097218 to find out who. POBox 6, LS15HB 150p",
    "You have an important customer service announcement from PREMIER.",
]

# Pick fifteen additional spam messages to serve as the discovery sample
remaining = sms[(sms.label == "spam") & (~sms["text"].isin(TARGET_SPAM))]["text"].tolist()
random.shuffle(remaining)
DISCOVERY_SPAM = remaining[:15]

heldout = set(TARGET_SPAM + DISCOVERY_SPAM)
sms = sms[~sms["text"].isin(heldout)]

spam_msgs = sms.loc[sms.label == "spam", "text"].tolist()
ham_msgs = sms.loc[sms.label == "ham", "text"].tolist()
random.shuffle(spam_msgs)
random.shuffle(ham_msgs)

print(f"Training pool: {len(spam_msgs)} spam + {len(ham_msgs)} ham messages")
print(f"{len(TARGET_SPAM)} attack targets + {len(DISCOVERY_SPAM)} discovery messages held out")

Training pool: 722 spam + 4825 ham messages
6 attack targets + 15 discovery messages held out


## The Target

Here we train a MultinomialNB model on the training data. From this point on, `clf` and `vec` are never touched again. The attack only ever sees a simple `query(text) → spam_probability` function.

In [3]:
X = spam_msgs + ham_msgs
y = ["spam"] * len(spam_msgs) + ["ham"] * len(ham_msgs)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=1, stratify=y)

vec = CountVectorizer(stop_words="english")
clf = MultinomialNB(alpha=1.0).fit(vec.fit_transform(X_train), y_train)
print(f"Target trained: {clf.score(vec.transform(X_test), y_test):.1%} held-out accuracy, "
      f"{len(vec.get_feature_names_out())} vocabulary words")

Target trained: 98.1% held-out accuracy, 7738 vocabulary words


## Candidate Vocabulary

Since we can't see inside the model, we have no way to know ahead of time which words it favors. So I picked a list of likely candidates from general knowledge: casual chat words, common SMS slang, words that just come up a lot in everyday texting.

I picked around fifty words by hand so the notebook is self-contained. In a real engagement, I'd gather this list from a bigger public dataset, but the method would be the same.

The list includes a few words on purpose that shouldn't work well ("go", "anyway", "true"), so the search actually has to tell good words from bad ones. A few of these ("go", "see", "please") are on scikit-learn's default stop-word list, meaning `CountVectorizer` throws them away before the model even sees them. So their real effect is exactly zero, not just small, and the search still has to work that out on its own, same as a real attacker would. I also left out words that the white-box notebook showed to be top performers ("lor", "da"), since a real attacker wouldn't know those in advance.

In [4]:
CANDIDATES = [
    # casual acknowledgments
    "ok", "okay", "sure", "yeah", "yep", "cool", "nice", "fine", "great", "right",
    # conversational fillers
    "lol", "gud", "cos", "haha", "hey", "hi", "hello",
    # temporal
    "later", "morning", "tonight", "tomorrow", "today", "weekend",
    # verbs of interaction
    "come", "go", "see", "say", "said", "talk", "chat", "ask", "tell",
    "doing", "thinking", "feel", "want", "need",
    # relationship
    "miss", "love", "friend", "mate", "dear",
    # fillers / discourse markers
    "way", "true", "anyway",
    # politeness
    "please", "thanks", "welcome", "sorry",
]
CANDIDATES = list(dict.fromkeys(CANDIDATES))
print(f"Candidate pool: {len(CANDIDATES)} words")

Candidate pool: 49 words


## Query Oracle

Every query has a cost. On a real API that cost might be a rate limit, a bill, or the risk of getting noticed. Here, it's just a counter that stops us once we hit a set limit.

The oracle remembers past answers. If we send the exact same text twice, the second time is free. This matters because both UCB and the final attack step often end up testing the same word-plus-message combination more than once, and the cache makes the repeat ones free.

There's one catch specific to this notebook: the full scan below tests the exact same word-and-message pairs that UCB would also want to test. If they shared one oracle, running the scan first would make UCB's results look cheaper than they really are, since UCB would just be reading cached answers. So the scan gets its own separate oracle (it's only here for comparison; no real attacker would ever run a full scan), and the real attack uses a separate budget. UCB discovery, the pair-testing step, and the final attack all share that second oracle, so queries they genuinely reuse between each other are fine, but none of it is propped up by the scan.

In [5]:
class QueryOracle:
    '''A budgeted black-box oracle over a text-scoring function.'''

    def __init__(self, query_fn, budget):
        self._fn = query_fn
        self._budget = budget
        self._used = 0
        self._cache = {}
        self._history = []

    def query(self, text):
        if text in self._cache:
            return self._cache[text]
        if self._used >= self._budget:
            raise RuntimeError(f"Budget exhausted ({self._budget} queries)")
        prob = float(self._fn(text))
        self._used += 1
        self._cache[text] = prob
        self._history.append((text, prob))
        return prob

    def used(self):
        return self._used

    def remaining(self):
        return self._budget - self._used


# ============ BLACK-BOX BOUNDARY ============
# Below this line the notebook uses only oracle.query(). It never
# touches clf, vec, feature_log_prob_, or the training data.
# ============================================


spam_idx = list(clf.classes_).index("spam")

def _target_query(text):
    '''Stand-in for the remote /predict endpoint. The attack never sees
    this function's implementation -- only the numeric output.'''
    return clf.predict_proba(vec.transform([text]))[0, spam_idx]


# scan_oracle: reference-only exhaustive baseline, used purely so this
# notebook can report how close UCB gets to the ideal ranking. A real
# attacker would never run this.
#
# atk_oracle: the one budget a real attacker actually has. Discovery,
# the synergy check, and final assembly all draw from it, so queries get
# legitimately reused across those stages -- but none of them get a free
# ride off scan_oracle's cache.
scan_oracle = QueryOracle(_target_query, budget=2000)
atk_oracle = QueryOracle(_target_query, budget=1000)
print("QueryOracles ready (scan_oracle: reference-only; atk_oracle: real attack budget).")

QueryOracles ready (scan_oracle: reference-only; atk_oracle: real attack budget).


## Baseline: Full Scan

This is the simplest way to search: try every word on every message, measure how much the spam score drops, average it, and rank the words. With 15 messages and 49 words, that's 735 tests plus 15 baseline checks, so 750 queries total. It's expensive, but it's thorough: every word gets tested fifteen times, so the ranking is reliable.

For this kind of model, that thorough ranking is close to the best possible answer. That's why it's useful as the thing we compare the other, cheaper methods against.

In [6]:
def full_scan(oracle, discovery_msgs, candidates):
    baselines = {m: oracle.query(m) for m in discovery_msgs}
    rows = []
    for word in candidates:
        rewards = []
        for msg in discovery_msgs:
            prob = oracle.query(msg + " " + word)
            rewards.append(baselines[msg] - prob)
        rows.append({"word": word, "mean_reward": np.mean(rewards), "std": np.std(rewards)})
    return pd.DataFrame(rows).sort_values("mean_reward", ascending=False).reset_index(drop=True)


start = scan_oracle.used()
scan_df = full_scan(scan_oracle, DISCOVERY_SPAM, CANDIDATES)
scan_used = scan_oracle.used() - start

print(f"Full scan complete ({scan_used} queries, reference-only oracle).")
print("Top 10 words by mean reward:")
for _, row in scan_df.head(10).iterrows():
    print(f"  {row['word']:<15} reward: {row['mean_reward']:.4f}")

Full scan complete (750 queries, reference-only oracle).
Top 10 words by mean reward:
  later           reward: 0.1163
  come            reward: 0.1126
  said            reward: 0.1083
  doing           reward: 0.1081
  ask             reward: 0.1081
  morning         reward: 0.1050
  cos             reward: 0.1047
  lol             reward: 0.1029
  sure            reward: 0.1025
  nice            reward: 0.1001


## UCB1 Discovery

The full scan is the ideal case, but it only works when the budget is much bigger than `words × messages`. On a real target, with a rate-limited API, hundreds of candidate words, and only a handful of messages to test on, a full scan just isn't affordable. UCB does the same job with a smaller budget, by being picky about which word-and-message pair to test next.

Each word is treated as an arm. Words we haven't tried yet get an infinite score, so each gets tried at least once. After that, each word's score is `mean_reward + c·sqrt(ln(t) / n)`, and whichever word scores highest gets tested next. The reward is the same thing we measured in the full scan: how much the spam probability dropped, compared to that message's starting point.

The `pulls_budget` setting below counts how many times UCB tries a word, not how many real queries that costs. Since some of those tries land on messages or words we've already tested, a few come back free from the cache. The print statements further down always show the real, after-cache number, which is why 400 tries only ends up costing 338 real queries.

In [7]:
class Arm:
    def __init__(self, word):
        self.word = word
        self.n = 0
        self.total = 0.0

    def update(self, reward):
        self.n += 1
        self.total += reward

    @property
    def mean(self):
        return self.total / self.n if self.n > 0 else 0.0

    def ucb_score(self, total_pulls, c):
        if self.n == 0:
            return float("inf")
        return self.mean + c * np.sqrt(np.log(max(total_pulls, 2)) / self.n)


def ucb_discover(oracle, discovery_msgs, candidates, pulls_budget, c=2.0):
    arms = [Arm(w) for w in candidates]
    baselines = {m: oracle.query(m) for m in discovery_msgs}
    pulls = 0
    while pulls < pulls_budget:
        total = sum(a.n for a in arms)
        arm = max(arms, key=lambda a: a.ucb_score(total, c))
        msg = discovery_msgs[pulls % len(discovery_msgs)]
        prob = oracle.query(msg + " " + arm.word)
        arm.update(baselines[msg] - prob)
        pulls += 1
    df = pd.DataFrame([{"word": a.word, "mean_reward": a.mean, "pulls": a.n} for a in arms])
    return df.sort_values("mean_reward", ascending=False).reset_index(drop=True)


start = atk_oracle.used()
ucb_df = ucb_discover(atk_oracle, DISCOVERY_SPAM, CANDIDATES, pulls_budget=400)
ucb_used = atk_oracle.used() - start

print(f"UCB discovery complete ({ucb_used} real queries against the attack budget).")
print("Top 10 words by mean reward:")
for _, row in ucb_df.head(10).iterrows():
    print(f"  {row['word']:<15} reward: {row['mean_reward']:.4f}")

UCB discovery complete (338 real queries against the attack budget).
Top 10 words by mean reward:
  cos             reward: 0.2478
  ok              reward: 0.1609
  okay            reward: 0.1569
  come            reward: 0.1516
  fine            reward: 0.1196
  tomorrow        reward: 0.1173
  later           reward: 0.1009
  said            reward: 0.0994
  lol             reward: 0.0993
  doing           reward: 0.0938


### How close did UCB come?

UCB uses its own separate oracle (`atk_oracle`), not the scan's (`scan_oracle`), so this comparison is fair: no shared cache, no free head start. Here's the real overlap and the real query cost.

In [8]:
scan_top10 = set(scan_df.head(10)["word"])
ucb_top10 = set(ucb_df.head(10)["word"])
overlap = len(scan_top10 & ucb_top10)

print(f"Top-10 overlap: {overlap}/10")
print(f"Query cost: scan={scan_used}, UCB={ucb_used} ({100 * (1 - ucb_used / scan_used):.0f}% saving)")
print(f"Pull distribution across UCB arms: min={ucb_df['pulls'].min()}, "
      f"max={ucb_df['pulls'].max()}, median={int(ucb_df['pulls'].median())}")

Top-10 overlap: 6/10
Query cost: scan=750, UCB=338 (55% saving)
Pull distribution across UCB arms: min=7, max=10, median=8


## Combination Search

Testing words one at a time only catches effects that work on their own. Some models(e.g. Transformer classifiers) reward certain word pairs more than you'd expect just from adding up their separate effects. That's called synergy. This model (MultinomialNB) doesn't have that. It just adds up log-odds for each word, so `impact(w1 + w2)` is always exactly `impact(w1) + impact(w2)`. That's just how this kind of model works, not a weakness in the search method.

The code below tests word pairs from the top of the UCB list and checks the gap between what we'd expect (if effects just add up) and what we actually see. For this model, that gap should land right around zero. On a transformer, or a model with feature crosses, the same code would show pairs with a real positive gap, meaning the pair genuinely works better together. That's the case where testing pairs is actually worth the extra queries.

I measure impact in log-odds here, not probability. Probability can be misleading near 0% and 100%, because of how the sigmoid curve squashes values out there (the same issue mentioned above for the UCB rewards, just stronger with two words instead of one).

In [9]:
def log_odds(p):
    p = min(max(p, 1e-10), 1 - 1e-10)
    return float(np.log(p / (1 - p)))


def measure_synergy(oracle, base, top_words, max_pairs=20):
    baseline_lo = log_odds(oracle.query(base))
    solo = {}
    for w in top_words:
        solo[w] = baseline_lo - log_odds(oracle.query(base + " " + w))

    rows = []
    all_pairs = list(combinations(top_words, 2))
    random.shuffle(all_pairs)
    for w1, w2 in all_pairs[:max_pairs]:
        joint = baseline_lo - log_odds(oracle.query(base + " " + w1 + " " + w2))
        expected = solo[w1] + solo[w2]
        rows.append({
            "pair": f"{w1} + {w2}",
            "solo_a": round(solo[w1], 4),
            "solo_b": round(solo[w2], 4),
            "expected": round(expected, 4),
            "actual": round(joint, 4),
            "synergy": round(joint - expected, 4),
        })
    return pd.DataFrame(rows).sort_values("synergy", ascending=False)


top10_words = list(ucb_df.head(10)["word"])
syn_start = atk_oracle.used()
synergy_df = measure_synergy(atk_oracle, TARGET_SPAM[0], top10_words, max_pairs=20)
synergy_used = atk_oracle.used() - syn_start
print(f"Top 10 pairs tested on a single target message ({synergy_used} real queries):")
synergy_df.head(5)

Top 10 pairs tested on a single target message (31 real queries):


,pair,solo_a,solo_b,expected,actual,synergy
0,cos + tomorrow,0.0,0.0,0.0,0.0,0.0
1,okay + tomorrow,0.0,0.0,0.0,0.0,0.0
18,ok + come,0.0,0.0,0.0,0.0,0.0
17,ok + fine,0.0,0.0,0.0,0.0,0.0
16,cos + come,0.0,0.0,0.0,0.0,0.0


The synergy column comes out at zero for every pair, exactly as expected. For this kind of model, `log-odds(w1 + w2)` is always exactly `log-odds(w1) + log-odds(w2)`, so there's nothing extra for the search to find. That's a useful result even though it's "nothing happened": it proves the code works correctly, and it tells us not to waste queries testing pairs against a model like this one. Against a model where pairs really do interact, this same column would show real numbers, and the best pair would go straight into the attack.

## The Attack

Now that we have a ranked list of words from UCB, the final step is the same as in the white-box notebook: add words one at a time, strongest first, and stop as soon as the message flips to "ham." The only real difference is that our ranking comes from testing, not from reading weights, so every check here also costs a query.

In [10]:
def attack_with_ranking(oracle, target, ranked_words, word_budget, threshold=0.5):
    added = []
    baseline = oracle.query(target)
    for w in ranked_words:
        if len(added) >= word_budget:
            break
        added.append(w)
        prob = oracle.query(target + " " + " ".join(added))
        if prob < threshold:
            return {"added": added, "baseline": baseline, "prob": prob, "flipped": True}
    final_prob = oracle.query(target + " " + " ".join(added)) if added else baseline
    return {"added": added, "baseline": baseline, "prob": final_prob, "flipped": final_prob < threshold}


WORD_BUDGET = 25
ranked_words = list(ucb_df["word"])
asm_start = atk_oracle.used()
results = [attack_with_ranking(atk_oracle, m, ranked_words, WORD_BUDGET) for m in TARGET_SPAM]
assembly_used = atk_oracle.used() - asm_start

table = pd.DataFrame([{
    "Original (truncated)": TARGET_SPAM[i][:55] + ("…" if len(TARGET_SPAM[i]) > 55 else ""),
    "Words added": " ".join(r["added"]),
    "# added": len(r["added"]),
    "Before": f"spam ({r['baseline']:.0%})",
    "After": f"{'ham' if r['flipped'] else 'spam'} ({(1 - r['prob']) if r['flipped'] else r['prob']:.0%})",
    "Flipped": "yes" if r["flipped"] else "NO",
} for i, r in enumerate(results)])

print(f"{sum(r['flipped'] for r in results)}/{len(results)} messages flipped to ham, "
      f"using an average of {np.mean([len(r['added']) for r in results]):.1f} added words "
      f"({assembly_used} real assembly queries).")
table

6/6 messages flipped to ham, using an average of 7.5 added words (49 real assembly queries).


,Original (truncated),Words added,# added,Before,After,Flipped
0,Free entry in 2 a wkly comp to win FA Cup final tkts 21…,cos ok okay come fine tomorrow later said lol doing say hey,12,spam (100%),ham (67%),yes
1,U have won a nokia 6230 plus a free digital camera. Thi…,cos ok okay come fine tomorrow later said lol doing,10,spam (100%),ham (93%),yes
2,"Thanks for your ringtone order, reference number X29. Y…",cos ok okay come fine tomorrow later said,8,spam (100%),ham (97%),yes
3,Want a new Video Phone? 750 anytime any network mins? H…,cos ok okay come fine tomorrow later said lol doing,10,spam (100%),ham (95%),yes
4,We know someone who you know that fancies you. Call 090…,cos ok,2,spam (100%),ham (58%),yes
5,You have an important customer service announcement fro…,cos ok okay,3,spam (100%),ham (76%),yes


In [11]:
ex = results[0]
print("=== Example ===\n")
print("BEFORE:", TARGET_SPAM[0])
print("\nAFTER: ", TARGET_SPAM[0] + " " + " ".join(ex["added"]))
ham_conf = (1 - ex['prob']) if ex['flipped'] else ex['prob']
print(f"\nFlipped from spam ({ex['baseline']:.0%}) to ham ({ham_conf:.0%}) with "
      f"{len(ex['added'])} words, using {len(ex['added']) + 1} assembly queries.")
print(f"\nReal query cost against the attack budget (atk_oracle, budget={atk_oracle._budget}):")
print(f"  UCB discovery:   {ucb_used}")
print(f"  Synergy check:   {synergy_used}")
print(f"  Assembly:        {assembly_used}")
print(f"  Total used:      {atk_oracle.used()} / {atk_oracle._budget} "
      f"({atk_oracle.remaining()} remaining)")
print(f"\nFor comparison, the reference-only exhaustive scan spent {scan_used} queries "
      f"on its own separate budget. A real attacker would never run it; it exists "
      f"here purely so this notebook can report how close UCB gets to the ideal ranking.")

=== Example ===

BEFORE: Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's

AFTER:  Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's cos ok okay come fine tomorrow later said lol doing say hey

Flipped from spam (100%) to ham (67%) with 12 words, using 13 assembly queries.

Real query cost against the attack budget (atk_oracle, budget=1000):
  UCB discovery:   338
  Synergy check:   31
  Assembly:        49
  Total used:      418 / 1000 (582 remaining)

For comparison, the reference-only exhaustive scan spent 750 queries on its own separate budget. A real attacker would never run it; it exists here purely so this notebook can report how close UCB gets to the ideal ranking.


## Where This Wouldn't Work

- **Label-only APIs.** If the API only tells you "spam" or "ham," with no probability, this gets much harder. You can still tell if a word flipped the result, but not how close it got. On a message that starts at 99% spam, one word almost never flips it, so most single-word tests just come back "spam" and tell you nothing useful. You'd end up having to test whole payloads instead of single words, which costs way more queries.
- **Rate limits and pricing tiers.** UCB here assumes a fixed number of queries. Real APIs often make early queries cheap and later ones expensive, or cap bursts of requests. Under that kind of pricing, you'd want a version of UCB that accounts for cost, not the plain version used here.
- **Models that don't just add things up.** This whole attack depends on the model treating each word as an independent vote. A transformer that reads the whole message together doesn't work that way, so a pile of random-looking words at the end often just gets flagged as low quality. You could still find words that flip one specific message, but a payload that works on one message likely won't transfer to another.
- **Coherence checks.** If the target also runs a second check for whether the message makes sense, or looks for a string of odd words at the end, this kind of padding gets caught. You'd have to blend the good words into the message instead of just appending them, which is a much harder problem.
- **Running out of budget.** If the query budget is smaller than the number of candidate words, no search method can reliably work. The budget has to be bigger than the candidate pool for the search to have a chance. UCB just helps you use a tight budget more efficiently. It can't make up for a budget that's too small to begin with.

The real weakness in a bag-of-words classifier isn't in its specific numbers. It's that bag-of-words scoring treats a whole message as just a pile of separate word-votes, with no sense of whether the message as a whole makes sense. Every method in this notebook takes advantage of that one fact in a different way. UCB just makes it cheap to find the right words under a tight budget.

That's also why just hiding the model doesn't protect much. Even without seeing any weights, an attacker with a few hundred queries can work out which words to use. The query budget is really the main defense, and it has to be tight enough that this kind of search can't finish. In practice, most production APIs set their limits based on normal user traffic, which is usually far more generous than the few hundred queries this attack actually needs.

## Defences

- Don't return a raw probability. A label-only response is always harder for an attacker to use than one with a probability attached. If you do need to show confidence, use broad buckets ("very likely spam", "likely spam", "uncertain") instead of an exact number.
- Rate-limit hard, per account, per IP, and per API key. A 100-query budget per session makes this kind of search basically impossible. A 1000-query budget doesn't.
- Move past pure bag-of-words scoring. N-gram features, models that read context, or a second model that checks coherence all break the assumption this attack depends on.
- Add a simple check for padding or word salad. A cheap rule, like flagging messages where the last few words are all strongly "ham" but don't fit the sentence, catches most of these attacks without needing a full second model.
- Train the model on examples of this exact attack, so it learns to discount bolted-on padding even when it's present.

## References

- Lowd, D. & Meek, C. (2005) - [Good Word Attacks on Statistical Spam Filters](https://personal.utdallas.edu/~mxk055100/courses/dmsec_files/125.pdf), CEAS 2005
- Auer, P., Cesa-Bianchi, N. & Fischer, P. (2002) - [Finite-time Analysis of the Multiarmed Bandit Problem, Machine Learning 47:235-256](https://link.springer.com/article/10.1023/A:1013689704352)
- Almeida, T.A., Gomez Hidalgo, J.M. & Yamakami, A. (2011) - [Contributions to the Study of SMS Spam Filtering](https://archive.ics.uci.edu/dataset/228/sms+spam+collection), DOCENG'11
- MITRE ATLAS - [AML.T0015, Evade AI Model](https://d3fend.mitre.org/offensive-technique/attack/AML.T0015/)
- MITRE ATLAS - [AML.T0043, Craft Adversarial Data](https://d3fend.mitre.org/offensive-technique/attack/AML.T0043/)
- OWASP ML Security Top 10 - [ML01:2023, Input Manipulation Attack](https://owasp.org/www-project-machine-learning-security-top-10/docs/ML01_2023-Input_Manipulation_Attack)
